# ACT-Africa 2026: Bearing Challenge

**Can a model diagnose a bearing it has never seen?**

You saw the answer in the lecture: balanced accuracy **0.554** for the frozen foundation model across twelve bearings, working on seven of them and failing on five.

Your objective is **not** simply a bigger number:

> *Can you improve unseen-bearing diagnosis while keeping the LOBO evaluation correct and
> explain why your change works?*

Note before you start: the classical-feature reference scores **0.634** under the identical protocol, so switching feature sets clears 0.554 in one line. Say which reference you are comparing against.

There is one evaluation protocol and it is the one from the lecture — leave-one-bearing-out over the same twelve bearings, same metric, same unit. All reported challenge results produced by lobo_score use the same 12-bearing LOBO evaluation as the **"Generalization to Unseen Bearings"** lecture slide.

Everything runs on a laptop CPU with no internet. Work down the page in order.
| | |
|---|---|
| **1** | What you have |
| **2** | The protocol — run it once, unchanged |
| **3** | The baselines, and the one that should surprise you |
| **4** | Read the per-bearing table |
| **5** | Your model |

## The protocol gate

*Only a number from lobo_score counts as a hackathon result, and your first slide has to show verify protocol -> PASSED. Without it the submission is invalid and we will not score it at all*.

**`train_test_split`** will not be accepted here. Each bearing gives you roughly 600 windows that are near-duplicates of one another, so a random split puts the same physical bearing in training and in testing, and the model can exploit bearing-specific information rather than demonstrating generalization to an unseen bearing. The numbers on these embeddings: **0.863** for a stratified 75/25 random split, **0.554** under the correct protocol. That gap is the point of the whole lecture.

**Fit learned preprocessing inside the fold.** Every mean, standard deviation, PCA basis, whitening matrix, feature selection step and tuned threshold sees the eleven training bearings and nothing else. Putting the step inside the **Pipeline** handles this for you, because **lobo_score calls** `clone(model).fit(X_train, y_train)` once per fold. Fit on the whole pool first and you have leakage that lobo_score cannot detect: 0.543 for the leaky version against 0.564 for the correct one. You will not spot the difference by reading the score.


## 1. What you have

In [1]:
import numpy as np, pandas as pd
from evaluate import (lobo_score, verify_protocol, report, pool_mask,
                      POOL, POOL_BEARINGS, BASELINES, CLASS_NAMES)

d = np.load("data/challenge.npz", allow_pickle=True)
pool = pool_mask(d["bearing"])          # the 12 protocol bearings

print("arrays:", d.files)
print(f"\n{len(d['y']):>5} windows from {len(set(d['bearing']))} bearings in the file")
print(f"{pool.sum():>5} windows from {len(POOL_BEARINGS)} bearings in the protocol pool")
print(f"{(~pool).sum():>5} windows from the 4 machined-damage bearings (training material only)")
print("\nthe pool, by class:")
for c, bs in POOL.items():
    print(f"  {CLASS_NAMES[c]:<18} {bs}")

arrays: ['embeddings', 'classical', 'envelope', 'y', 'bearing', 'recording', 'origin', 'condition']

 9624 windows from 16 bearings in the file
 7209 windows from 12 bearings in the protocol pool
 2415 windows from the 4 machined-damage bearings (training material only)

the pool, by class:
  Healthy            ['K001', 'K002', 'K003', 'K004']
  Outer-race fault   ['KA04', 'KA15', 'KA16', 'KA22']
  Inner-race fault   ['KI04', 'KI14', 'KI16', 'KI17']


Three views of the same 256 ms of vibration. Use any of them, or anything you derive.

| array | width | what it is |
|---|---|---|
| `embeddings` | 512 | the frozen foundation model's description of the window |
| `classical` | 17 | RMS, kurtosis, crest factor, and envelope-spectrum energy at BPFO and BPFI harmonics |
| `envelope` | 512 | the demodulated window itself, before any model touches it |

The pack holds **16 bearings**: 4 healthy, 8 with real damage from accelerated lifetime tests, and 4 whose damage was machined in (KA01, KA03, KI01, KI03 by electric discharge machining or an electric engraver). The evaluation pool is the 4 healthy plus the 8 real-damage bearings. The 4 machined bearings are never an evaluation fold; they are extra training material, and whether they help is a question worth answering.

Documented damage for the eight: seven are fatigue pitting; **KA15 is plastic deformation with indentations**, a different mechanism. Severity varies too KI16 is level 3, KA16 level 2, the rest level 1. The healthy bearings differ in run-in time (50, 19, 1 and 5 hours). All of these are covariates you can use when explaining which bearings fail.

In [2]:
X   = {k: d[k] for k in ["embeddings", "classical", "envelope"]}
y, bearing, recording = d["y"], d["bearing"], d["recording"]

print("windows and recordings per pool bearing:")
for b in POOL_BEARINGS:
    m = bearing == b
    print(f"  {b:6s} class {y[m][0]}   {m.sum():4d} windows   {len(set(recording[m])):2d} recordings")

windows and recordings per pool bearing:
  K001   class 0    600 windows   20 recordings
  K002   class 0    602 windows   20 recordings
  K003   class 0    600 windows   20 recordings
  K004   class 0    600 windows   20 recordings
  KA04   class 1    600 windows   20 recordings
  KA15   class 1    600 windows   20 recordings
  KA16   class 1    600 windows   20 recordings
  KA22   class 1    602 windows   20 recordings
  KI04   class 2    600 windows   20 recordings
  KI14   class 2    603 windows   20 recordings
  KI16   class 2    602 windows   20 recordings
  KI17   class 2    600 windows   20 recordings


## 2. The protocol

12 folds. Hold out one whole bearing, train on the other **11**, predict the held-out one. Repeat for all **12**, pool the predictions, score once.

**Check it before you trust it.** "What did you hold out" is the first thing on the judging sheet.

In [3]:
ok = verify_protocol(bearing[pool], recording[pool])
assert ok, "fix the split before going further"

PROTOCOL CHECK: PASSED
  12 folds, one bearing held out each time
  no recording spans a fold boundary
  7209 windows over 240 recordings


Why leave-one-bearing-out rather than a single train/test split? Because there are only four bearings per class. We measured every possible way of holding out one bearing per class the score lands anywhere between **0.07 and 0.97** depending purely on which three you pick. Rotating through all twelve removes that luck. A single split here would be a lottery, not an experiment.

## 3. The baselines

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def probe(C=1.0):
    return Pipeline([("scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                C=C, random_state=42))])

# 12 folds per feature set. Takes about a minute in total - it is not stuck.
res = {}
for name in ["envelope", "classical", "embeddings"]:
    res[name] = lobo_score(probe(), X[name][pool], y[pool], bearing[pool], verbose=False)
    print(f"  {name} done")

print()
print(pd.DataFrame([{"features": k,
                     "balanced_accuracy": round(v["balanced_accuracy"], 3),
                     "macro_f1": round(v["macro_f1"], 3)} for k, v in res.items()]
                   ).to_string(index=False))
print("\nchance = 0.333")

  envelope done
  classical done
  embeddings done

  features  balanced_accuracy  macro_f1
  envelope              0.323     0.322
 classical              0.633     0.637
embeddings              0.554     0.555

chance = 0.333


In [5]:
#check with the RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.linear_model import LogisticRegression

def probe(C=1.0):
    return Pipeline([("scale", RobustScaler()),
                     ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                C=C, random_state=42))])

# 12 folds per feature set. Takes about a minute in total - it is not stuck.
res = {}
for name in ["envelope", "classical", "embeddings"]:
    res[name] = lobo_score(probe(), X[name][pool], y[pool], bearing[pool], verbose=False)
    print(f"  {name} done")

print()
print(pd.DataFrame([{"features": k,
                     "balanced_accuracy": round(v["balanced_accuracy"], 3),
                     "macro_f1": round(v["macro_f1"], 3)} for k, v in res.items()]
                   ).to_string(index=False))
print("\nchance = 0.333")


  envelope done
  classical done
  embeddings done

  features  balanced_accuracy  macro_f1
  envelope              0.323     0.322
 classical              0.653     0.657
embeddings              0.557     0.558

chance = 0.333


Using the same classifier and the same 12 fold LOBO evaluation, classical features achieve 0.634 balanced accuracy, compared with 0.554 for frozen MOMENT embeddings and 0.323 for the envelope baseline (chance = 0.333).

Under this unseen-bearing evaluation, the classical feature representation therefore performs better than the frozen MOMENT representation. This result applies to this specific dataset and evaluation setting and should not be interpreted as a general comparison between handcrafted features and foundation models.

The classical features encode bearing-specific information, including BPFO/BPFI harmonic content, whereas the frozen MOMENT embeddings provide a general time-series representation. The following experiments test whether stronger regularisation or additional machined-damage training data can improve the MOMENT result.

In [6]:
# about 50 seconds
for C in [0.001, 0.01]:
    r = lobo_score(probe(C), X["embeddings"][pool], y[pool], bearing[pool], verbose=False)
    print(f"MOMENT, C = {C:<7} balanced accuracy {r['balanced_accuracy']:.3f}")
print(f"MOMENT, C = {1.0:<7} balanced accuracy {res['embeddings']['balanced_accuracy']:.3f}   (from above)")

MOMENT, C = 0.001   balanced accuracy 0.583
MOMENT, C = 0.01    balanced accuracy 0.570
MOMENT, C = 1.0     balanced accuracy 0.557   (from above)


### Do the machined-damage bearings help?

They are not a test fold, but they are 2,415 extra labelled windows. `extra_train` adds them
to **every** training fold and never tests on them, so the protocol is unchanged and the
number stays comparable.

In [7]:
base  = res["embeddings"]                      # already computed above
withA = lobo_score(probe(), X["embeddings"][pool], y[pool], bearing[pool],
                   extra_train=(X["embeddings"][~pool], y[~pool], bearing[~pool]),
                   verbose=False)
print(f"MOMENT alone                    {base['balanced_accuracy']:.3f}")
print(f"MOMENT + machined in training   {withA['balanced_accuracy']:.3f}")
print(f"                         delta  {withA['balanced_accuracy']-base['balanced_accuracy']:+.3f}")

MOMENT alone                    0.557
MOMENT + machined in training   0.590
                         delta  +0.033


## 4. The per-bearing table

The pooled number hides the finding. Look at the bearings individually.

In [8]:
r = res["embeddings"]                          # already computed above
_ = report(y[pool], r["oof"], r["per_bearing"], "MOMENT frozen, leave-one-bearing-out")


MOMENT frozen, leave-one-bearing-out
balanced accuracy 0.5568

                           Healthy  Outer-race f  Inner-race f     recall
Healthy                       1279           762           361      0.532
Outer-race fault               839          1171           392      0.488
Inner-race fault               373           468          1564      0.650
rows = truth, columns = prediction

per-bearing recall, worst last - this is the table that matters
  K003   1.000  ##############################
  K001   0.995  ##############################
  KI04   0.965  #############################
  KA04   0.942  ############################
  KA16   0.938  ############################
  KI16   0.817  #########################
  KI17   0.687  #####################
  KI14   0.134  ####
  K002   0.128  ####
  KA22   0.068  ##
  K004   0.008  
  KA15   0.003  

  mean 0.557   sd 0.422   worst 0.003 on KA15


Seven bearings score above 0.60 and five below 0.15, with nothing in between. The model therefore succeeds on some bearings and fails almost completely on others. The pooled balanced accuracy of 0.554 hides this strong bearing-to-bearing variation.

The model does not work moderately well everywhere. It works on some bearings and fails almost completely on others, and the pooled balanced accuracy of 0.554 describes none of them. If you report only the pooled number tomorrow you will have hidden your most interesting result.

Questions worth an answer:

- What do KA15, K004 and KA22 have in common that KA04 and KA16 do not? (KA15 is the one
  bearing whose damage is plastic deformation rather than fatigue pitting but that is a
  hypothesis with a sample size of one, not an explanation.)
- Is a failing bearing being pushed into one particular wrong class, or scattered?
- Would you rather have a model at 0.554 that fails on five bearings, or one at 0.50 that
  gets every bearing to 0.50? Which would a maintenance team prefer, and why?

## 5. Your model

You may explore different features, classifiers, regularisation, feature combinations, a reject option, the machined bearings as extra training data, and explicitly declared adaptation methods while preserving the grouped LOBO evaluation protocol.

Three rules, and they are the ones you are judged on:

1. **Score with `lobo_score` and nothing else.** A number from any other protocol is not
   comparable to the lecture, the references, or the other teams, and is not a valid
   submission. `train_test_split` in particular is invalid — see the protocol gate at the
   top of this notebook.
2. **Do not rebuild the folds.** The protocol is fixed so that every team's number means
   the same thing.
3. **Fit every preprocessing parameter on the training bearings only.**

> **Leakage rule.** Any preprocessing parameter learned from data — a mean, a standard
> deviation, a PCA basis, a whitening matrix — must be fitted on the **training bearings
> only**, inside the fold, unless you explicitly declare your method as target adaptation
> and report it separately from your LOBO number. The supplied `probe()` already does this
> correctly: the `StandardScaler` sits inside the `Pipeline`, so `clone(model).fit(X_train,
> y_train)` fits it on the eleven training bearings and merely applies it to the held-out
> one.

One idea this rules out, and it is worth knowing why. Standardising each bearing using that bearing's own windows looks harmless you always know which machine a signal came from. But inside a fold it uses the held-out bearing's own distribution before predicting it, which is transductive rather than inductive. And it does not even help here: each bearing carries exactly one class, so centring per bearing removes the class signal too. Under this protocol it gives **0.333** exactly chance, with every healthy bearing at recall 1.0 and every damaged bearing at 0.0. If you want target adaptation, declare it explicitly and report it separately from your LOBO number.

**This is a transparent development benchmark, not a blinded final test set.** You can see the score while you change your model, so your final number carries some selection bias by construction exactly as it would for a researcher tuning against cross-validation. Nothing removes that in an afternoon. What makes your result trustworthy is that the search is visible, so record what you tried.

Ideas that are worth the afternoon:

- **Strong regularisation.** 512 dimensions, 11 training bearings. See section 3.
- **Reduce nuisance variation the safe way.** If between-bearing variation is hurting you,
  put the transform — a PCA, a whitening step — inside the `Pipeline`, where `clone().fit()`
  will fit it on the eleven training bearings for you.
- **Combine embeddings with classical features.** The classical features carry the physics;
  the embeddings may carry something they miss.
- **A reject option.** Abstain when the top two probabilities are close, and report coverage
  alongside accuracy. A model that says "I don't know" on the five hard bearings is more
  deployable than one that guesses confidently.
- **Fix one bearing.** Pick KA15 and work out what would have to be true for it to work.

In [9]:
# ------------------------------------------------------------------ No change, used as it was given and then i define my own cells next
features = "embeddings"          # or "classical", or something you build
model    = probe()
extra    = None                  # or (X[features][~pool], y[~pool], bearing[~pool])
                                 # to add the machined bearings to every training fold
# -------------------------------------------------------------------------------

r = lobo_score(model, X[features][pool], y[pool], bearing[pool], extra_train=extra)
_ = report(y[pool], r["oof"], r["per_bearing"], f"your model on {features}")


balanced accuracy  0.557      (baseline 0.554 · chance 0.333)
macro F1           0.558
per-bearing recall  7 above 0.60 · 5 below 0.15 · 0 in between

your model on embeddings
balanced accuracy 0.5568

                           Healthy  Outer-race f  Inner-race f     recall
Healthy                       1279           762           361      0.532
Outer-race fault               839          1171           392      0.488
Inner-race fault               373           468          1564      0.650
rows = truth, columns = prediction

per-bearing recall, worst last - this is the table that matters
  K003   1.000  ##############################
  K001   0.995  ##############################
  KI04   0.965  #############################
  KA04   0.942  ############################
  KA16   0.938  ############################
  KI16   0.817  #########################
  KI17   0.687  #####################
  KI14   0.134  ####
  K002   0.128  ####
  KA22   0.068  ##
  K004   0.008  
  KA15   0.003

In [10]:
#Start with the Robust scaller [explanation and changes made are needed]
# ---------------------------------------------------------------------------
# WHAT CHANGED FROM THE BASELINE probe():
# ---------------------------------------------------------------------------
from sklearn.linear_model import LogisticRegression


#   1. Features changed to combined: classical (17) + embeddings (512) concatenated, instead of just one set to reduce the noise/spacity
X["combined"] = np.hstack([X["classical"], X["embeddings"]])   # physics + learned rep

#   2. Classifier: L1 penalty instead of default L2, so it can zero out useless dimensions
def probe_l1(C=0.05): #from probe() to a new function
    return Pipeline([
        ("scale", RobustScaler()),  # 4. Scaler: Changed from standardScaler
        ("clf",   LogisticRegression(penalty="l1", solver="liblinear",
                                      max_iter=2000, class_weight="balanced",
                                      C=C, random_state=42)),
    ])

# ------------------------------------------------------------------ our model
features = "combined"
model    = probe_l1(C=0.05)

#   3. Training data: machined bearings added via extra_train (already proven +0.036 in section 3)
extra    = (X[features][~pool], y[~pool], bearing[~pool])   # machined bearings — already proven +0.036
# -------------------------------------------------------------------------------

r = lobo_score(model, X[features][pool], y[pool], bearing[pool], extra_train=extra)
_ = report(y[pool], r["oof"], r["per_bearing"], f"your model on {features}")

balanced accuracy  0.676      (baseline 0.554 · chance 0.333)
macro F1           0.675
per-bearing recall  8 above 0.60 · 2 below 0.15 · 2 in between

your model on combined
balanced accuracy 0.6762

                           Healthy  Outer-race f  Inner-race f     recall
Healthy                       1802           230           370      0.750
Outer-race fault               593          1339           470      0.557
Inner-race fault                20           651          1734      0.721
rows = truth, columns = prediction

per-bearing recall, worst last - this is the table that matters
  K001   1.000  ##############################
  K003   1.000  ##############################
  KA04   1.000  ##############################
  KI04   1.000  ##############################
  K002   0.990  ##############################
  KA16   0.965  #############################
  KI16   0.787  ########################
  KI17   0.683  ####################
  KI14   0.415  ############
  KA22   0.216  

In [11]:
#Change the scaler [explanation is needed]
from sklearn.linear_model import LogisticRegression

X["combined"] = np.hstack([X["classical"], X["embeddings"]])   # physics + learned rep

def probe_l1(C=0.05):
    return Pipeline([
        ("scale", StandardScaler()),
        ("clf",   LogisticRegression(penalty="l1", solver="liblinear",
                                      max_iter=2000, class_weight="balanced",
                                      C=C, random_state=42)),
    ])

# ------------------------------------------------------------------ your model
features = "combined"
model    = probe_l1(C=0.05)
extra    = (X[features][~pool], y[~pool], bearing[~pool])   # machined bearings — already proven +0.036
# -------------------------------------------------------------------------------

r = lobo_score(model, X[features][pool], y[pool], bearing[pool], extra_train=extra)
_ = report(y[pool], r["oof"], r["per_bearing"], f"your model on {features}")

balanced accuracy  0.679      (baseline 0.554 · chance 0.333)
macro F1           0.677
per-bearing recall  8 above 0.60 · 2 below 0.15 · 2 in between

your model on combined
balanced accuracy 0.6789

                           Healthy  Outer-race f  Inner-race f     recall
Healthy                       1800           250           352      0.749
Outer-race fault               567          1338           497      0.557
Inner-race fault                 9           640          1756      0.730
rows = truth, columns = prediction

per-bearing recall, worst last - this is the table that matters
  K001   1.000  ##############################
  K003   1.000  ##############################
  KA04   1.000  ##############################
  KI04   1.000  ##############################
  K002   0.995  ##############################
  KA16   0.963  #############################
  KI16   0.824  #########################
  KI17   0.700  #####################
  KI14   0.398  ############
  KA22   0.218

In [12]:
# ---------------------------------------------------------------------------
# CHANGED: instead of one fixed C=0.05, sweep several C values in a loop and
# report all results together, so the search is visible (not just the winner)
# per the "record what you tried" judging rule.
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# CHANGED: C grid now starts at 0.001, matching the sweep already run on
# embeddings alone in section 3 (C=0.001 gave 0.578, C=0.01 gave 0.568,
# C=1.0 gave 0.554). Reusing the same starting points keeps this comparable
# to that earlier experiment rather than introducing an unrelated grid.
# ---------------------------------------------------------------------------

#C_values = [0.01, 0.05, 0.1, 0.5]   # small = strong regularization as already seen in teh baseline + foudnation mode code above
C_values = [0.001, 0.01, 0.05, 0.1, 0.5]   # 0.001/0.01 reused from section 3; 0.05/0.1/0.5 extend the search

results_l1 = {}

for C in C_values:
    r_c = lobo_score(
        probe_l1(C=C),
        X["combined"][pool], y[pool], bearing[pool],
        extra_train=(X["combined"][~pool], y[~pool], bearing[~pool]),
        verbose=False
    )
    results_l1[C] = r_c
    print(f"  C={C:<6} balanced accuracy {r_c['balanced_accuracy']:.3f}")

# summarize as a table, same style as the baseline comparison in section 3
print()
print(pd.DataFrame([{"C": c,
                     "balanced_accuracy": round(v["balanced_accuracy"], 3),
                     "macro_f1": round(v["macro_f1"], 3)} for c, v in results_l1.items()]
                   ).to_string(index=False))

# pick the best C by score, but keep the FULL table above visible in your slide —
# that's the "search is visible" requirement, not just the winning number
best_C = max(results_l1, key=lambda c: results_l1[c]["balanced_accuracy"])
print(f"\nbest C = {best_C}  (balanced accuracy {results_l1[best_C]['balanced_accuracy']:.3f})")

# carry the best one forward for the per-bearing report
r = results_l1[best_C]
_ = report(y[pool], r["oof"], r["per_bearing"], f"your model on combined, C={best_C}")

  C=0.001  balanced accuracy 0.623
  C=0.01   balanced accuracy 0.703
  C=0.05   balanced accuracy 0.679
  C=0.1    balanced accuracy 0.672
  C=0.5    balanced accuracy 0.680

    C  balanced_accuracy  macro_f1
0.001              0.623     0.625
0.010              0.703     0.702
0.050              0.679     0.677
0.100              0.672     0.670
0.500              0.680     0.679

best C = 0.01  (balanced accuracy 0.703)

your model on combined, C=0.01
balanced accuracy 0.7027

                           Healthy  Outer-race f  Inner-race f     recall
Healthy                       1801           164           437      0.750
Outer-race fault               461          1457           484      0.607
Inner-race fault                61           536          1808      0.752
rows = truth, columns = prediction

per-bearing recall, worst last - this is the table that matters
  K001   1.000  ##############################
  K003   1.000  ##############################
  KA04   1.000  ########

In [13]:
#the output above; Classical: 9/17 (53%) kept. L1 retained roughly half the hand-engineered features
#Embeddings: 37/512 (7%) kept. Out of 512 learned dimensions, only 37 survived regularization.

#This calls for trying other classifiers;

# Matched comparison: same feature sets, multiple classifiers, same protocol
# Answers whether embeddings need a non-linear model to be useful, or just don't help regardless

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

def probe_l2(C=1.0):              # baseline logistic regression (linear)
    return Pipeline([("scale", RobustScaler()),
                      ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                  C=C, random_state=42))])

def probe_rf(n_estimators=300, max_depth=6):   # random forest (non-linear)
    return Pipeline([("scale", RobustScaler()),
                      ("clf", RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                                       class_weight="balanced", random_state=42))])

def probe_svm(C=1.0, gamma="scale"):           # SVM with RBF kernel (non-linear)
    return Pipeline([("scale", RobustScaler()),
                      ("clf", SVC(C=C, gamma=gamma, class_weight="balanced", random_state=42))])

classifiers = {
    "logistic_l2":   probe_l2(),
    "random_forest": probe_rf(),
    "svm_rbf":       probe_svm(),
}

results_multi = {}
for clf_name, probe_fn in classifiers.items():
    for feat in ["classical", "embeddings", "combined"]:
        rr = lobo_score(probe_fn, X[feat][pool], y[pool], bearing[pool], verbose=False)
        results_multi[(clf_name, feat)] = rr["balanced_accuracy"]
        print(f"  {clf_name:14s} {feat:10s} balanced accuracy {rr['balanced_accuracy']:.3f}")

# summary table, sorted best first
print()
df = pd.DataFrame([{"classifier": k[0], "features": k[1], "balanced_accuracy": round(v, 3)}
                    for k, v in results_multi.items()]).sort_values("balanced_accuracy", ascending=False)
print(df.to_string(index=False))

  logistic_l2    classical  balanced accuracy 0.653
  logistic_l2    embeddings balanced accuracy 0.557
  logistic_l2    combined   balanced accuracy 0.650
  random_forest  classical  balanced accuracy 0.687
  random_forest  embeddings balanced accuracy 0.583
  random_forest  combined   balanced accuracy 0.594
  svm_rbf        classical  balanced accuracy 0.766
  svm_rbf        embeddings balanced accuracy 0.567
  svm_rbf        combined   balanced accuracy 0.637

   classifier   features  balanced_accuracy
      svm_rbf  classical              0.766
random_forest  classical              0.687
  logistic_l2  classical              0.653
  logistic_l2   combined              0.650
      svm_rbf   combined              0.637
random_forest   combined              0.594
random_forest embeddings              0.583
      svm_rbf embeddings              0.567
  logistic_l2 embeddings              0.557


In [14]:
# Best classical model and best embeddings model so far — compare per-bearing, not just pooled
from sklearn.svm import SVC
for C in [0.1, 1.0, 10.0]:
    for gamma in ["scale", 0.01, 0.1]:
        rr = lobo_score(probe_svm(C=C, gamma=gamma), X["classical"][pool], y[pool], bearing[pool], verbose=False)
        print(f"  C={C:<6} gamma={gamma!s:<6} balanced accuracy {rr['balanced_accuracy']:.3f}")

  C=0.1    gamma=scale  balanced accuracy 0.747
  C=0.1    gamma=0.01   balanced accuracy 0.657
  C=0.1    gamma=0.1    balanced accuracy 0.723
  C=1.0    gamma=scale  balanced accuracy 0.766
  C=1.0    gamma=0.01   balanced accuracy 0.741
  C=1.0    gamma=0.1    balanced accuracy 0.730
  C=10.0   gamma=scale  balanced accuracy 0.768
  C=10.0   gamma=0.01   balanced accuracy 0.772
  C=10.0   gamma=0.1    balanced accuracy 0.738


In [15]:
r_best = lobo_score(probe_svm(C=10.0, gamma=0.01), X["classical"][pool], y[pool], bearing[pool], verbose=False)
_ = report(y[pool], r_best["oof"], r_best["per_bearing"], "classical, SVM C=10, gamma=0.01")


classical, SVM C=10, gamma=0.01
balanced accuracy 0.7717

                           Healthy  Outer-race f  Inner-race f     recall
Healthy                       1798           600             4      0.749
Outer-race fault                70          1945           387      0.810
Inner-race fault                 2           583          1820      0.757
rows = truth, columns = prediction

per-bearing recall, worst last - this is the table that matters
  K001   1.000  ##############################
  K003   1.000  ##############################
  KI04   1.000  ##############################
  KI16   1.000  ##############################
  K002   0.993  ##############################
  KI17   0.987  ##############################
  KA16   0.973  #############################
  KA22   0.900  ###########################
  KA04   0.868  ##########################
  KA15   0.497  ###############
  KI14   0.043  #
  K004   0.000  

  mean 0.772   sd 0.362   worst 0.000 on K004


In [16]:
# #THIS FAILURE ON K004!

# mask_k004 = bearing[pool] == "K004"
# preds_k004 = r_best["oof"][mask_k004]
# print(pd.Series(preds_k004).value_counts())

In [17]:
# # FORM ABOVE RESULT all 600 windows of K004 are predicted as class 1 (Outer-race fault),
# # unanimously. every single window lands on the exact same wrong class

# # SO WE NEED TO pull K004's actual BPFO-energy value (one of the 17 classical features) and compare it to the other three healthy bearings:

# bpfo_idx = 0  # replace with the actual column index for BPFO energy in your classical feature array — check evaluate.py or feature docs
# for b in ["K001", "K002", "K003", "K004"]:
#     m = (bearing == b)
#     print(f"  {b}: mean BPFO-energy feature = {X['classical'][m][:, bpfo_idx].mean():.4f}")

In [18]:
# import matplotlib.pyplot as plt
# import numpy as np

# # ============================================================
# # FIGURE 1: Per-bearing recall, baseline vs final model
# # Uses res["embeddings"]["per_bearing"] (baseline) and
# # r_best["per_bearing"] (your tuned SVM on classical) — both
# # already computed earlier in the notebook.
# # If per_bearing is a dict {bearing: recall}, this works as-is;
# # if it's structured differently, adjust the two lookups below.
# # ============================================================

# baseline_pb = res["embeddings"]["per_bearing"]
# final_pb    = r_best["per_bearing"]

# bearings = sorted(baseline_pb, key=lambda b: baseline_pb[b])  # worst baseline first
# old_vals = [baseline_pb[b] for b in bearings]
# new_vals = [final_pb[b]    for b in bearings]

# fig, ax = plt.subplots(figsize=(8, 6))
# y_pos = np.arange(len(bearings))

# for y, old, new in zip(y_pos, old_vals, new_vals):
#     color = "#639922" if new >= 0.6 else "#E24B4A"   # green if rescued, red if still failing
#     ax.plot([old, new], [y, y], color="#888780", zorder=1, linewidth=1.5)
#     ax.scatter(old, y, color="#888780", zorder=2, s=50, label="Baseline" if y == 0 else "")
#     ax.scatter(new, y, color=color, zorder=2, s=60, label="Final model" if y == 0 else "")

# ax.axvline(0.6, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
# ax.text(0.6, len(bearings) - 0.3, "0.60 threshold", fontsize=9, ha="center", color="gray")

# ax.set_yticks(y_pos)
# ax.set_yticklabels(bearings)
# ax.set_xlabel("Recall")
# ax.set_xlim(-0.02, 1.05)
# ax.set_title("Per-bearing recall: baseline (embeddings) vs final model (classical + SVM)")
# ax.legend(loc="lower right")
# ax.spines[["top", "right"]].set_visible(False)
# plt.tight_layout()
# plt.savefig("per_bearing_recovery.png", dpi=150)
# plt.show()

# # ============================================================
# # FIGURE 2: Accuracy growth across experiments, by change type
# # Edit `steps` to match your actual attempts list if it differs.
# # ============================================================

# steps = [
#     ("Embeddings\nbaseline",  0.554, "#888780"),  # gray  = starting point
#     ("Regularize\nC=0.001",   0.578, "#378ADD"),  # blue  = regularization
#     ("+ Machined\nbearings",  0.590, "#1D9E75"),  # teal  = training data
#     ("Combine\nfeatures",     0.650, "#7F77DD"),  # purple = feature engineering
#     ("Switch to\nSVM",        0.766, "#D85A30"),  # coral = classifier choice
#     ("Tune\nSVM",             0.772, "#D85A30"),
# ]

# labels = [s[0] for s in steps]
# scores = [s[1] for s in steps]
# colors = [s[2] for s in steps]

# fig, ax = plt.subplots(figsize=(9, 5))
# x = np.arange(len(steps))

# ax.plot(x, scores, color="#888780", linewidth=1.5, zorder=1)
# ax.scatter(x, scores, c=colors, s=90, zorder=2, edgecolor="white", linewidth=1)

# for xi, s in zip(x, scores):
#     ax.annotate(f"{s:.3f}", (xi, s), textcoords="offset points", xytext=(0, 10),
#                 ha="center", fontsize=10, fontweight="bold")

# ax.axhline(0.333, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
# ax.text(-0.3, 0.333, "chance = 0.333", fontsize=9, va="bottom", color="gray")
# ax.axhline(0.634, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
# ax.text(-0.3, 0.634, "classical baseline = 0.634", fontsize=9, va="bottom", color="gray")

# ax.set_xticks(x)
# ax.set_xticklabels(labels)
# ax.set_ylabel("Balanced accuracy")
# ax.set_ylim(0.25, 0.85)
# ax.set_title("Balanced accuracy across experiments, colored by what changed")
# ax.spines[["top", "right"]].set_visible(False)
# plt.tight_layout()
# plt.savefig("accuracy_growth.png", dpi=150)
# plt.show()

In [20]:
# Was MOMENT Needed?
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as LR

Xtr, Xte, btr, bte = train_test_split(X["embeddings"][pool], bearing[pool], test_size=0.3,
                                       stratify=bearing[pool], random_state=42)
clf = Pipeline([("scale", RobustScaler()), ("clf", LR(max_iter=2000))]).fit(Xtr, btr)
print(f"Bearing-identity accuracy from embeddings: {clf.score(Xte, bte):.3f}")

Bearing-identity accuracy from embeddings: 0.816


In [ ]:
# MOMENT was not needed as the bearing identity is high.

#frozen MOMENT embeddings are highly informative 81.6% accurate, about which specific bearing produced a signal, 
#but only weakly informative (55–59%) about what fault that bearing has. 
#The representation encodes bearing identity, not fault physics, which is precisely the wrong bias for a leave-one-bearing-out generalization task.